<a href="https://colab.research.google.com/github/love-wedding/ba_phi-hong_nhung/blob/master/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

# 1. Setup (~1 minute)

In [2]:
import os
import subprocess
import torch

assert torch.cuda.is_available(), "❌ GPU not detected! Runtime -> Change runtime type -> T4 GPU"

print("GPU :", torch.cuda.get_device_name(0))
print("Torch :", torch.__version__)

# Clone nếu chưa có
if not os.path.exists("Real-ESRGAN"):
    subprocess.run(
        ["git", "clone", "https://github.com/xinntao/Real-ESRGAN.git"],
        check=True
    )

%cd Real-ESRGAN

# Update pip
!python -m pip install -q --upgrade pip

# Không cài lại torch (dùng torch có sẵn của Colab)
!pip install -q -r requirements.txt

# Cài các package cần thiết
!pip install -q facexlib gfpgan ffmpeg-python

# Cài project
!python setup.py develop

print("✅ Setup completed.")

GPU : Tesla T4
Torch : 2.11.0+cu128
/content/Real-ESRGAN/Real-ESRGAN
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.1 MB/s eta 0:00:00
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/b


# 2. Mount drive (optional)

In [4]:
from google.colab import drive
mount_drive=False #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

In [10]:
from google.colab import files
import os

uploaded = files.upload()

filename = next(iter(uploaded))

# Lấy đúng đường dẫn tuyệt đối của file vừa upload
video_path = os.path.abspath(filename)

print("Video:", video_path)
print("Exists:", os.path.exists(video_path))

Saving output.mp4 to output.mp4
Video: /content/Real-ESRGAN/Real-ESRGAN/output.mp4
Exists: True


In [8]:
import os

print(os.getcwd())
print(os.listdir())

/content/Real-ESRGAN/Real-ESRGAN
['docs', 'LICENSE', 'setup.cfg', 'weights', 'inference_realesrgan.py', 'realesrgan', 'Use_the_first_uploaded_image_202607132205.mp4', 'experiments', 'Use_the_first_uploaded_image_202607132205 (1).mp4', 'VERSION', 'setup.py', 'requirements.txt', '.github', '.gitignore', 'inputs', 'cog_predict.py', 'README_CN.md', '.pre-commit-config.yaml', '.vscode', 'scripts', 'realesrgan.egg-info', 'inference_realesrgan_video.py', 'options', 'MANIFEST.in', '.git', 'assets', 'CODE_OF_CONDUCT.md', 'README.md', 'cog.yaml', 'tests']


In [17]:
!pip install -q torchvision==0.20.1

# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [11]:
import os
import cv2
import subprocess

# Nếu chưa chạy Cell 3 thì mới dùng đường dẫn nhập tay
try:
    video_path
except NameError:
    video_path = "/content/video.mp4"  #@param {type:"string"}

output_dir = "/content/"  #@param {type:"string"}
resolution = "FHD (1920 x 1080)"  #@param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)", "2 x original", "3 x original", "4 x original"]
model = "RealESRGAN_x4plus"  #@param ["RealESRGAN_x4plus","RealESRGAN_x4plus_anime_6B","realesr-animevideov3"]

assert os.path.exists(video_path), f"❌ Video file does not exist:\n{video_path}"

video_capture = cv2.VideoCapture(video_path)

video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_capture.release()

final_width = None
final_height = None
aspect_ratio = video_width / video_height

# Output resolution
if resolution == "FHD (1920 x 1080)":
    final_width = 1920
    final_height = 1080

elif resolution == "2k (2560 x 1440)":
    final_width = 2560
    final_height = 1440

elif resolution == "4k (3840 x 2160)":
    final_width = 3840
    final_height = 2160

elif resolution == "2 x original":
    final_width = video_width * 2
    final_height = video_height * 2

elif resolution == "3 x original":
    final_width = video_width * 3
    final_height = video_height * 3

elif resolution == "4 x original":
    final_width = video_width * 4
    final_height = video_height * 4

# Video vuông
if aspect_ratio == 1 and "original" not in resolution:
    final_height = final_width

# Video dọc
if aspect_ratio < 1 and "original" not in resolution:
    final_width, final_height = final_height, final_width

scale_factor = max(
    final_width / video_width,
    final_height / video_height
)

while (
    int(video_width * scale_factor) % 2 != 0
    or int(video_height * scale_factor) % 2 != 0
):
    scale_factor += 0.01

print("=" * 60)
print("Input :", video_path)
print(f"Original : {video_width}x{video_height}")
print(f"Output   : {final_width}x{final_height}")
print(f"Scale    : {scale_factor:.2f}")
print("=" * 60)

# Upscale
!python inference_realesrgan_video.py \
    -n {model} \
    -i "{video_path}" \
    -o "{output_dir}" \
    --outscale {scale_factor}

video_name = os.path.splitext(os.path.basename(video_path))[0]

upscaled_video_path = os.path.join(
    output_dir,
    f"{video_name}_out.mp4"
)

final_video_name = f"{video_name}_upscaled_{final_width}_{final_height}.mp4"

final_video_path = os.path.join(
    output_dir,
    final_video_name
)

# Crop về đúng kích thước nếu chọn FHD/2K/4K
if "original" not in resolution:

    print("Cropping...")

    command = f"""
    ffmpeg -loglevel error -y \
    -i "{upscaled_video_path}" \
    -filter:v "crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2" \
    -c:v libx264 \
    -pix_fmt yuv420p \
    "{final_video_path}"
    """

    subprocess.run(command, shell=True)

else:

    subprocess.run(
        f'cp "{upscaled_video_path}" "{final_video_path}"',
        shell=True
    )

print()
print("✅ Done!")
print("Saved:", final_video_path)

# Lưu sang Google Drive nếu đã mount
if mount_drive:

    drive_folder = "/content/gdrive/MyDrive/Upscaled Videos (REAL-ESRGAN)"
    os.makedirs(drive_folder, exist_ok=True)

    subprocess.run(
        f'cp "{final_video_path}" "{drive_folder}/{final_video_name}"',
        shell=True
    )

    print("Saved to Google Drive.")

# Xóa file tạm
if os.path.exists(upscaled_video_path):
    os.remove(upscaled_video_path)

Input : /content/Real-ESRGAN/Real-ESRGAN/output.mp4
Original : 720x1280
Output   : 1080x1920
Scale    : 1.50
Traceback (most recent call last):
  File "/content/Real-ESRGAN/Real-ESRGAN/inference_realesrgan_video.py", line 10, in <module>
    from basicsr.archs.rrdbnet_arch import RRDBNet
  File "/usr/local/lib/python3.12/dist-packages/basicsr/__init__.py", line 4, in <module>
    from .data import *
  File "/usr/local/lib/python3.12/dist-packages/basicsr/data/__init__.py", line 22, in <module>
    _dataset_modules = [importlib.import_module(f'basicsr.data.{file_name}') for file_name in dataset_filenames]
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/basicsr/data/realesrgan_dataset.py", line 11, in <module>

In [13]:
from google.colab import files
import os

# Kiểm tra file tồn tại
if os.path.exists(final_video_path):
    print("📥 Downloading:", final_video_path)
    files.download(final_video_path)
else:
    print("❌ Không tìm thấy file:", final_video_path)

❌ Không tìm thấy file: /content/output_upscaled_1080_1920.mp4


In [20]:
import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

ImportError: cannot import name '_HAS_OPS' from 'torchvision.extension' (/usr/local/lib/python3.12/dist-packages/torchvision/extension.py)

# 4. Disconnect runtime

In [ ]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()